<a href="https://colab.research.google.com/github/Kishoby/Conceptual-Research_Hybrid-Approach/blob/Temperature_Celsius/Temperature_Statistical.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np

from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.api import VAR

from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Load your datasets
train_df = pd.read_csv("/content/drive/MyDrive/Research v2/Research 18.03.2026/Temperature_Celsius/temp_training_dataset.csv")
test_df = pd.read_csv("/content/drive/MyDrive/Research v2/Research 18.03.2026/Temperature_Celsius/temp_testing_dataset.csv")

target = "temperature_celsius"

# ⚠️ If you have datetime column (recommended)
# If your column name is "last_updated", use this:
# train_df["last_updated"] = pd.to_datetime(train_df["last_updated"])
# test_df["last_updated"] = pd.to_datetime(test_df["last_updated"])

# train_df.set_index("last_updated", inplace=True)
# test_df.set_index("last_updated", inplace=True)

# Evaluation function
def evaluate(y_true, y_pred):
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)

    y_true_safe = np.where(np.array(y_true) == 0, 1e-10, y_true)
    acc = 100 - (np.mean(np.abs((y_true - y_pred) / y_true_safe)) * 100)

    return mse, rmse, mae, r2, acc

In [ ]:
import pandas as pd
import numpy as np
from statsmodels.tsa.arima.model import ARIMA
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Load datasets
train_df = pd.read_csv("/content/drive/MyDrive/Research v2/Research 18.03.2026/Temperature_Celsius/temp_training_dataset.csv")
test_df = pd.read_csv("/content/drive/MyDrive/Research v2/Research 18.03.2026/Temperature_Celsius/temp_testing_dataset.csv")

target = "temperature_celsius"

# Keep only target and make numeric
train_target = pd.to_numeric(train_df[target], errors="coerce").dropna()
test_target = pd.to_numeric(test_df[target], errors="coerce").dropna()

print("Train target length:", len(train_target))
print("Test target length:", len(test_target))

# Evaluation function
def evaluate(y_true, y_pred):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)

    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)

    y_true_safe = np.where(y_true == 0, 1e-10, y_true)
    accuracy = 100 - (np.mean(np.abs((y_true - y_pred) / y_true_safe)) * 100)

    return mse, rmse, mae, r2, accuracy

# Fit ARIMA
model = ARIMA(train_target, order=(5, 1, 0))
model_fit = model.fit()

# Forecast exactly test length
forecast = model_fit.forecast(steps=len(test_target))

print("Forecast length:", len(forecast))

# Make sure both lengths match
min_len = min(len(test_target), len(forecast))
test_target = test_target.iloc[:min_len]
forecast = np.array(forecast)[:min_len]

metrics_arima = evaluate(test_target, forecast)

print("\nARIMA Results")
print("MSE:", metrics_arima[0])
print("RMSE:", metrics_arima[1])
print("MAE:", metrics_arima[2])
print("R2:", metrics_arima[3])
print("Accuracy (%):", metrics_arima[4])

Train target length: 104002
Test target length: 26001
Forecast length: 26001

ARIMA Results
MSE: 162.71480027989924
RMSE: 12.755971161769661
MAE: 9.392463429317823
R2: -0.3101279404658903
Accuracy (%): -49463599207.88602


In [ ]:
model = SARIMAX(
    train_df[target],
    order=(2,1,2),
    seasonal_order=(1,1,1,12)  # adjust if needed
)

model_fit = model.fit(disp=False)

forecast = model_fit.forecast(steps=len(test_df))

metrics_sarima = evaluate(test_df[target], forecast)

print("SARIMA Results")
print(metrics_sarima)

SARIMA Results
(157.82434411098106, np.float64(12.562815930792787), 9.308793290482983, -0.2707515391950652, np.float64(-48568659048.32197))


In [ ]:
print("SARIMA Results")

print("MSE:", metrics_sarima[0])
print("RMSE:", metrics_sarima[1])
print("MAE:", metrics_sarima[2])
print("R2:", metrics_sarima[3])
print("Accuracy (%):", metrics_sarima[4])

SARIMA Results
MSE: 157.82434411098106
RMSE: 12.562815930792787
MAE: 9.308793290482983
R2: -0.2707515391950652
Accuracy (%): -48568659048.32197


In [ ]:
var_features = [
    "temperature_celsius",
    "humidity",
    "pressure_mb",
    "air_quality_Ozone",
    "air_quality_Nitrogen_dioxide",
    "air_quality_PM10"
]

train_var = train_df[var_features].dropna()
test_var = test_df[var_features].dropna()

model = VAR(train_var)
model_fit = model.fit(maxlags=5)

lag_order = model_fit.k_ar

forecast = model_fit.forecast(
    train_var.values[-lag_order:],
    steps=len(test_var)
)

forecast_df = pd.DataFrame(
    forecast,
    index=test_var.index,
    columns=var_features
)

metrics_var = evaluate(
    test_var["temperature_celsius"],
    forecast_df["temperature_celsius"]
)

print("VAR Results")
print(metrics_var)

VAR Results
(164.0070448172982, np.float64(12.806523525816763), 9.415950263923214, -0.32053268343609664, np.float64(-49690346361.17258))


In [ ]:
results = pd.DataFrame([
    {
        "Model": "ARIMA",
        "MSE": metrics_arima[0],
        "RMSE": metrics_arima[1],
        "MAE": metrics_arima[2],
        "R2": metrics_arima[3],
        "Accuracy (%)": metrics_arima[4]
    },
    {
        "Model": "SARIMA",
        "MSE": metrics_sarima[0],
        "RMSE": metrics_sarima[1],
        "MAE": metrics_sarima[2],
        "R2": metrics_sarima[3],
        "Accuracy (%)": metrics_sarima[4]
    },
    {
        "Model": "VAR",
        "MSE": metrics_var[0],
        "RMSE": metrics_var[1],
        "MAE": metrics_var[2],
        "R2": metrics_var[3],
        "Accuracy (%)": metrics_var[4]
    }
]).round(4)

print(results)

    Model       MSE     RMSE     MAE      R2  Accuracy (%)
0   ARIMA  162.7148  12.7560  9.3925 -0.3101 -4.946360e+10
1  SARIMA  157.8243  12.5628  9.3088 -0.2708 -4.856866e+10
2     VAR  164.0070  12.8065  9.4160 -0.3205 -4.969035e+10


In [ ]:
import pandas as pd

stat_results_table = pd.DataFrame([
    {
        "Model": "ARIMA",
        "MSE": metrics_arima[0],
        "RMSE": metrics_arima[1],
        "MAE": metrics_arima[2],
        "R2": metrics_arima[3],
        "Accuracy (%)": metrics_arima[4]
    },
    {
        "Model": "SARIMA",
        "MSE": metrics_sarima[0],
        "RMSE": metrics_sarima[1],
        "MAE": metrics_sarima[2],
        "R2": metrics_sarima[3],
        "Accuracy (%)": metrics_sarima[4]
    },
    {
        "Model": "VAR",
        "MSE": metrics_var[0],
        "RMSE": metrics_var[1],
        "MAE": metrics_var[2],
        "R2": metrics_var[3],
        "Accuracy (%)": metrics_var[4]
    }
]).round(4)

print(stat_results_table)

    Model       MSE     RMSE     MAE      R2  Accuracy (%)
0   ARIMA  162.7148  12.7560  9.3925 -0.3101 -4.946360e+10
1  SARIMA  157.8243  12.5628  9.3088 -0.2708 -4.856866e+10
2     VAR  164.0070  12.8065  9.4160 -0.3205 -4.969035e+10


In [ ]:
stat_results_table.to_csv("statistical_model_results.csv", index=False)

print("Statistical results saved as CSV!")

Statistical results saved as CSV!


In [ ]:
from google.colab import files
files.download("statistical_model_results.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>